# LeCAD-CXR — Kaggle training runner

> **Runs on kaggle.com only.** Upload this file to Kaggle (File → Import Notebook). Do not run it in VS Code. For laptop checks, use the commands in Part A of `docs/training-guide.md`.

This notebook contains **no logic**. It fetches the repository, installs it, and calls
`ml/scripts/train.py` and `ml/scripts/evaluate.py` — the same scripts that run on the laptop.

Before running, in the right-hand panel:
1. **Session options → Accelerator:** a GPU.
2. **Session options → Internet:** on.
3. **Input:** the *NIH Chest X-rays* dataset is added.
4. **Add-ons → Secrets:** `GITHUB_TOKEN` exists and is ticked for this notebook.

Full walkthrough: `docs/training-guide.md` in the repository.

## 1. Settings — the only cell you edit

In [ ]:
# Which experiment to run: baseline.yaml | resnet50.yaml | efficientnet_b0.yaml
CONFIG = "ml/configs/baseline.yaml"

# Extra arguments for train.py.
#   Timing run (about 10-20 minutes):  "--limit 2000 --epochs 1"
#   Full run:                          ""
EXTRA = "--limit 2000 --epochs 1"

# To continue a run that was cut off, give the path of its folder inside an
# attached notebook output, e.g.
# "/kaggle/input/lecad-cxr-train/LeCAD-CXR/ml/artifacts/runs/
#  baseline-densenet121-20261006-101500"  (as one line, without the break)
# Leave empty for a fresh run.
RESUME_FROM = ""

## 2. Check the GPU and the dataset

In [ ]:
import glob
import os

if not os.path.isdir("/kaggle/input"):
    raise RuntimeError(
        "This notebook only runs on kaggle.com. You are running it locally. "
        "Upload it to Kaggle (File -> Import Notebook), or for laptop checks use "
        "Part A of docs/training-guide.md."
    )

!nvidia-smi -L

hits = (glob.glob("/kaggle/input/*/Data_Entry_2017.csv")
        + glob.glob("/kaggle/input/*/*/Data_Entry_2017.csv"))
assert hits, "Dataset not found. Add the 'NIH Chest X-rays' dataset under Input."
os.environ["LECAD_DATA_ROOT"] = os.path.dirname(hits[0])
print("data root:", os.environ["LECAD_DATA_ROOT"])

## 3. Fetch the code (private repo, token from Kaggle Secrets)

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "github.com/Kananelo12/LeCAD-CXR.git"

%cd /kaggle/working
!rm -rf LeCAD-CXR
!git clone -q https://{token}@{REPO} LeCAD-CXR
# Remove the token from the saved repo config: /kaggle/working becomes downloadable output.
!git -C LeCAD-CXR remote set-url origin https://{REPO}
del token

%cd /kaggle/working/LeCAD-CXR
!git log --oneline -1
!pip install -q -e ml

## 4. Train

In [ ]:
if RESUME_FROM:
    run_name = os.path.basename(RESUME_FROM.rstrip("/"))
    !mkdir -p ml/artifacts/runs
    !cp -r {RESUME_FROM} ml/artifacts/runs/
    EXTRA = f"--resume ml/artifacts/runs/{run_name}"

!python ml/scripts/train.py --config {CONFIG} {EXTRA}

## 5. Validation report for the best epoch

In [ ]:
run = max(glob.glob("ml/artifacts/runs/*"))
print("run:", run)
!python ml/scripts/evaluate.py --run {run}